# ISAAC controller client for Jupyter
This notebook is used for tests to create a fully functional ISAAC client that could complement the current web client.
Currently, it can be used for tasks like automatic responses to incoming messages or saving and restoring settings
between simulations.

TODO: image output

In [ ]:
from isaac_client import ProtocolHandler
#import ipywidgets as widgets
#from IPython import display
#import threading
import asyncio
import math

%gui asyncio

## Short explanation of `asyncio`

In order to concurrently receive, process and send messages from an to the server, the `asyncio` module is used.
This module introduces its own synthax to create and await asynchronous functions. 
The `ProtocolHandler` module that handles communication with ISAAC awaits for incoming messages. This means, it
switches back to the message receiver and processer task, once it receives messages.
To make the sending of messages non-blocking, they are also always awaited. 

One can send their own messages via `await app.send_feedback(message)`. If the logic for sending a message is included
in another function, the function must be made asyncronous (`async` keyword before definition) and also be calles with
`await` keyword.
Instead of `time.sleep()`, `await asyncio.sleep()` should be used, so control can be given back to other tasks in the
sleep time.

## Initialize client and observation

**Create connection to server**

The IP is the adress of the node of the server and should be checked with `ip addr` (or `ifconf` for classic users)
before starting the server.
It usually is the value under `lag_alsw99` and looks like `10.99.0.X`.

All functions that processed as part of the message receive task, print their output here.
By default, only messages for registration are printed. Set `verbose` to `True` to print all sent and received messages.
**Warning**: This might slow down the notebook significantly as the cell output fills up.

In [ ]:
# create handler object. IP should be changed (see above), the port can be left unchanged
app = ProtocolHandler("10.0.31.7", 2459, verbose=False)
# connect to server and start task that listens for incoming messages
await app.run_async_io()

**Start observation of stream**

The observe_id can be taken from the registration messages (see Visualization ID in output above).
Technically, one can observe multiple visualization at once, but this is not tested and all
incoming messages would be printed to the same output.

**Warning**: Currently, there is no check if the `observe_id` is valid. If the sim is not responding to feedback, you might be observing an invalid id.

Set `dropable` to True, if the delay between server and client gets too big.

In [ ]:
observe_id = 0
# stream id must be 2 for some reason?! TODO: Debug
await app.send_observe(observe_id, 2, dropable=False)

## End observation and connection

In [ ]:
# Send signal to stop observing current visualization
await app.send_stop()

In [ ]:
# Close connection to server
await app.send_closed()
# this would execute, even when send_closed is still in queue
#app.wsh.ws.close()

## Message sending example

Short example, on how to send messages to the server.

This will rotate the camera 45° around the $z$-axis.

Valid types of feedback can be found under https://computationalradiationphysics.github.io/isaac/doc/json/index.html

In [ ]:
# Rotation angle around z-axis (in rad) 
theta = math.pi / 4
# actually a nicely formated list
rotation_matrix = [math.cos(theta),  0, math.sin(theta),
                   0,                1, 0,
                   -math.sin(theta), 0, math.cos(theta)]
# must be awaited. Will be send, once asyncio handes it control over loop
await app.send_feedback({"rotation relative": rotation_matrix})

## Restoring settings

This part can be used to set parameters like the transfer functions and weight, so one does not have to send them by hand in the web client each time.

**Note**: It is recommended to only send this (and other feedback commands) after the first time step ran, else the simulation might crash.

**@Richard**: Currently, they are set to values that I find sensible for the TWEAC visualization but you can tweak them here or in the web client, if you want.

**Transfer functions**

Note, that the `value` is set in the range [0, 1024] (inclusive) and not [0,1] as in the web client.

**@Richard**: I changed the $E$-field to blue for negative values and red for positive ones. The $e$-density uses a yellow-orange-pink gradiant and a bit of green in the transition to the transparent part. The transition was chosen, so that the plasma is barely visible at rest and only become visible once the plasma is disturbed by the laser.

In [ ]:
transfer_EField =       [{'value': 0, 'r': 0.0, 'g': 0.0, 'b': 1.0, 'a': 1.0}, {'value': 511, 'r': 0.0, 'g': 0.0, 'b': 1.0, 'a': 0.0}, {'value': 513, 'r': 1.0, 'g': 0.0, 'b': 0.0, 'a': 0.0}, {'value': 1024, 'r': 1.0, 'g': 0.0, 'b': 0.0, 'a': 1.0}]
transfer_BField =       [{'value': 0, 'r': 1.0, 'g': 0.75, 'b': 0.0, 'a': 0.0}, {'value': 1024, 'r': 1.0, 'g': 0.75, 'b': 0.0, 'a': 1.0}]
transfer_JField =       [{'value': 0, 'r': 0.5, 'g': 1.0, 'b': 0.0, 'a': 0.0}, {'value': 1024, 'r': 0.5, 'g': 1.0, 'b': 0.0, 'a': 1.0}]
transfer_eDensity =     [{'value': 0, 'r': 0.0, 'g': 1.0, 'b': 0.25, 'a': 0.0}, {'value': 215, 'r': 0.0, 'g': 1.0, 'b': 0.25, 'a': 0.0}, {'value': 225, 'r': 1.0, 'g': 1.0, 'b': 0.25, 'a': 1.0}, {'value': 338, 'r': 1.0, 'g': 0.5, 'b': 0.0, 'a': 1.0}, {'value': 717, 'r': 1.0, 'g': 0.0, 'b': 0.8, 'a': 1.0}, {'value': 1024, 'r': 1.0, 'g': 0.6, 'b': 0.8, 'a': 1.0}]
transfer_EVectorField = [{'value': 0, 'r': 0.0, 'g': 1.0, 'b': 1.0, 'a': 0.0}, {'value': 1024, 'r': 0.0, 'g': 1.0, 'b': 1.0, 'a': 1.0}]
transfer_BVectorField = [{'value': 0, 'r': 0.0, 'g': 0.25, 'b': 1.0, 'a': 0.0}, {'value': 1024, 'r': 0.0, 'g': 0.25, 'b': 1.0, 'a': 1.0}]
transfer_JVectorField = [{'value': 0, 'r': 0.5, 'g': 0.0, 'b': 1.0, 'a': 0.0}, {'value': 1024, 'r': 0.5, 'g': 0.0, 'b': 1.0, 'a': 1.0}]
transfer_eParticle    = [{'value': 0, 'r': 1.0, 'g': 0.0, 'b': 0.75, 'a': 1.0}, {'value': 1024, 'r': 1.0, 'g': 0.0, 'b': 0.75, 'a': 1.0}]

transfer_points = [transfer_EField,
                   transfer_BField,
                   transfer_JField,
                   transfer_eDensity,
                   transfer_EVectorField,
                   transfer_BVectorField,
                   transfer_JVectorField,
                   transfer_eParticle
                  ]

**Functor chains**

**@Richard**: I only changed the $E$-fields to show the $E_y$-component and remap to [0,1].

In [ ]:
function_EField =       {'source': 'mul(0, 15, 0) | sum | add(0.5)', 'error': 0}
function_BField =       {'source': 'idem', 'error': 0}
function_JField =       {'source': 'idem', 'error': 0}
function_eDensity =     {'source': 'idem', 'error': 0}
function_EVectorField = {'source': 'length', 'error': 0}
function_BVectorField = {'source': 'length', 'error': 0}
function_JVectorField = {'source': 'length', 'error': 0}
function_eParticle =    {'source': 'idem', 'error': 0}

functions = [function_EField, function_BField, function_JField,
             function_eDensity, function_EVectorField, function_BVectorField,
             function_JVectorField, function_eParticle
            ]

**Weighting**

For volume and particle rendering.

In [ ]:
weight_EField =       0.0
weight_BField =       0.0
weight_JField =       0.0
weight_eDensity =     20.0
weight_EVectorField = 0.0 
weight_BVectorField = 0.0
weight_JVectorField = 0.0
weight_eParticle =    0.0

weight = [weight_EField, weight_BField, weight_JField, weight_eDensity,
          weight_EVectorField, weight_BVectorField, weight_JVectorField,
          weight_eParticle]

**ISO surface thresholds**

In [ ]:
iso_EField =       0.3
iso_BField =       0.0
iso_JField =       0.0
iso_eDensity =     0.0
iso_EVectorField = 0.0 
iso_BVectorField = 0.0
iso_JVectorField = 0.0

iso_threshold = [iso_EField, iso_BField, iso_JField, iso_eDensity,
                iso_EVectorField, iso_BVectorField, iso_JVectorField
                ]

**Miscellaneous settings**

In [ ]:
# ambient occlusion
ao_isEnabled = True
# interpolation for iso-surfaces
interpolation = True

Send updated values to ISAAC.

In [ ]:
await app.send_feedback({
    #"functions": functions, "weight": weight, "interpolation": interpolation,
    #"ao isEnabled": ao_isEnabled, "iso threshold": iso_threshold,
    "transfer points": transfer_points
})

## Response handler

In order to send commands on given timesteps, a response handler was added. This handler takes a function that will have the time step as an input, executes it upon receiving a message from the server and returns a feedback message that is immediatly send back to the server. This reduces the delay between the simulation reaching a time step and a response being send, which is important for things like automatic camera steering.

Currently, there is no setter for this response function. So, one still just overwrites the `app.response_handler` member with their own function. This can be done at any point during the simulation. If one wants to deactivate the handler, just set the member to `None`.

Note, that processor intensive calculations will block sending and receiving of messages and desync the client from the simulation.

**@Richard**: As an example, I included a handler that sets the camera rotation based on the time step. You can set the beginning and end time step and angles and the function will linearly interpolate between them. I hope, its self explanatory so you can change stuff like camera position and distance if needed.

Further below there is a cell that send rotations immediatly, so you can test different angles. 

I used absolut rotations, so that frame drops don't result in unpredictably slower movement.

In [ ]:
def lin_interpolate(x:float, x1:float, x2:float, y1:float, y2:float) -> float:
    """Linearly interpolate y at position x between point (x1,y1) and (x2,y2).
    Technically, this can also be used for extrapolating
    
    Parameters
    ----------
    x: float
        Position at which y is evaluated.
    x1, y1: float
        Starting positions.
    x2, y2: float
        End positions.

    Returns
    ----------
    y: float
        Interpolated value at x.
    """
    # Slope
    m = (y2 - y1) / (x2 - x1)
    y = m * (x - x1) + y1
    return y


def rotation_matrix(alpha:float, beta:float, gamma:float) -> list[float]:
    """Calculates flattened 3D rotation-matrix from given rotation angles.
    
    Parameters
    ----------
    alpha: float
        Roll, rotation around z-axis.
    beta: float
        Yaw, rotation around y-axis.
    gamma: float
        Pitch, rotation around x-axis.

    Returns
    ----------
    rotation_matrix: list[float]
        Flattened rotation matrix.
    """
    s_a = math.sin(alpha)
    c_a = math.cos(alpha)
    s_b = math.sin(beta)
    c_b = math.cos(beta)
    s_g = math.sin(gamma)
    c_g = math.cos(gamma)

    # formula for general 3D rotation matrix
    rotation_matrix = [c_a*c_b,  c_a*s_b*s_g - s_a*c_g, c_a*s_b*c_g + s_a*s_g,
                       s_a*c_b,  s_a*s_b*s_g + c_a*c_g, s_a*s_b*c_g - c_a*s_g,
                       -s_b,     c_b*s_g,               c_b*c_g              ]
    return rotation_matrix

In [ ]:
# TODO: Setting all parameters in the function itself is probably bad design
def camera_rotate(step: int) -> dict[str, list]:
    """Rotates the camera between given points based on the current time step
    via linear interpolation for the inbetween steps.
    
    Parameters
    ----------
    step: int
        Last received time step of visualization.
    
    Returns
    ----------
    response: dict[string, list]
        ISAAC message that contains the calculated rotation matrix for the
        current step as a list.        
    """

    # Start angles
    # roll: z-axis
    alpha_start = -math.pi/2
    # pitch: y-axis
    beta_start  =  math.pi/2
    # yaw: x-axis
    gamma_start =  math.pi/4

    # End angles
    alpha_end = -math.pi/2
    beta_end  =  0
    gamma_end =  0

    # Start and end step
    step_start  =  7800
    step_end  =  8800

    # Do nothing before start step or after end step
    # This still sends a message every step, so might it slow down a bit.
    if step < step_start:
        alpha = alpha_start
        beta  = beta_start
        gamma = gamma_start
    elif step > step_end:
        alpha = alpha_end
        beta  = beta_end
        gamma = gamma_end
    else:
        # get interpolated values
        alpha = lin_interpolate(step, step_start, step_end, alpha_start, alpha_end)
        beta  = lin_interpolate(step, step_start, step_end, beta_start, beta_end)
        gamma = lin_interpolate(step, step_start, step_end, gamma_start, gamma_end)

    rot_matrix = rotation_matrix(alpha, beta, gamma)
    response = {"rotation absolute": rot_matrix}
    #print(response)
    return response

In [ ]:
# Set handler
app.response_handler = camera_rotate

In [ ]:
# Remove handler
app.response_handler = None

Cell to test different angles immediately.

In [ ]:
# roll: z-axis
alpha_test = -math.pi/2
# pitch: y-axis
beta_test  = math.pi/2
# yaw: x-axis
gamma_test = math.pi/4

rotation_matrix_test = rotation_matrix(alpha_test, beta_test, gamma_test)

await app.send_feedback({"rotation absolute": rotation_matrix_test})

# Display tests (WIP)
Tests to get a working display output.
Currently, only the first received image is displayed and the display does not update.
Potential problems could be, that the image processing and output block to long and the receiver can't keep up with the
server anymore.

In [ ]:
#observer_text = widgets.Text(
#    value='0',
#    placeholder='Oberservation ID',
#    description='String:',
#    disabled=False   
#)
#display(observer_text)
observer_text = input()
# testing: connect to the some visualization
app.send_observe(observer_text, dropable=False)

In [ ]:
async def show_stream():
    while True:
        #image_data = display.Image(app.latest_image, format="jpeg", retina=True)
        #display.update_display(image_data, display_id=image_display)
        image_display.value = app.image_decoder(app.latest_image)
        observe_stream = observe_checkbox.value
        if not observe_stream:
            app.send_stop()
            return


In [ ]:
#image_data = display.Image(app.latest_image, format="jpeg", retina=True)

#image_display = display.display(image_data, display_id=True)
image_display = widgets.Image(value=app.image_decoder(app.latest_image))
observe_checkbox = widgets.Checkbox(
    value=True,
    description='Observe stream',
)
run_stream = observe_checkbox.value
#display_thread = threading.Thread(target=show_stream)
loop = asyncio.get_running_loop()
display_task = loop.create_task(show_stream())

display.display(image_display, observe_checkbox)

# 